In [ ]:
import subprocess
import sys

# N = 207, N0 = floor(pN), budget = N0^1.5, r = 3
# choose the largest k such that 1 + k + k^2 + k^3 <= N0^1.5
p_to_k = {
    "0.1": "4",
    "0.2": "6",
    "0.3": "7",
    "0.4": "8",
    "0.5": "9",
    "0.6": "10",
    "0.7": "11",
    "0.8": "12",
    "0.9": "13",
}

base_cmd = [
    sys.executable, "-u", "sfl_main_filtering.py",
    "--seed", "42",
    "--runs", "10",

    "--split-mode", "ratio",
    "--train-ratio", "0.7",

    "--truth-type", "hadamard",
    "--volterra-lambda", "10.0",
    "--volterra-quadratic-coeffs", "2,-4,2",
    "--nonlinear-scale", "0.0",

    "--noise-type", "skew_bernoulli",
    "--sigma", "20.0",

    "--loss-type", "smooth_pinball",
    "--huber-delta", "0.1",
    "--tau-pinball", "0.8",

    "--methods", "sub_lp,kron_lp,union_lp,dk_lap_poly,rg_alg,sffa,num_lmmse",

    "--r-sffa", "3",
    "--r-poly", "3",
    "--max-sffa-basis-size", "3000",

    "--optimizer", "lbfgs",
    "--epochs", "50",
    "--lbfgs-max-iter", "20",
    "--lr", "0.5",

    "--ridge", "0.",
    "--sfl-ridge", "0.",
]

for p_visible, k_sffa in p_to_k.items():
    print("\n" + "=" * 120, flush=True)
    print(f"Running p_visible={p_visible}, k_sffa={k_sffa}, r_sffa=3", flush=True)
    print("=" * 120, flush=True)

    cmd = base_cmd + [
        "--p-visible", p_visible,
        "--k-sffa", k_sffa,
        "--k-union", k_sffa,
        "--output-csv", f"filtering_summary_p{p_visible.replace('.', 'p')}.csv",
        "--output-raw-csv", f"filtering_raw_p{p_visible.replace('.', 'p')}.csv",
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end="", flush=True)

    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(
            f"Command failed for p_visible={p_visible}, k_sffa={k_sffa} "
            f"with exit code {ret}"
        )